# Phase 2 · Notebook 01 — HuggingFace Baseline (`dslim/bert-base-NER`)

In Phase 1 we used spaCy's strongest off-the-shelf English NER (`en_core_web_trf`) and got an overall partial-match F1 of ~0.57. The natural follow-up question is: **does swapping in a different general-purpose model help?**

[`dslim/bert-base-NER`](https://huggingface.co/dslim/bert-base-NER) is one of the most-downloaded NER models on the HuggingFace Hub. It's a `bert-base-cased` fine-tuned on CoNLL-2003 — so it predicts the four labels `PER`, `ORG`, `LOC`, `MISC`.

This notebook runs it through the same evaluation framework as Phase 1, against TAB's full test split. The expected finding: it'll close some of the spaCy gap on PERSON, but will be **even worse** on `DATETIME`, `QUANTITY`, `CODE`, and `DEM`, because those labels don't exist in CoNLL-2003. That's the empirical answer to *"can we just use a different general-purpose model?"*

---


In [1]:
# ── Run me first if you're on Colab (skip locally — already in requirements.txt) ──
# !pip -q install transformers datasets evaluate seqeval accelerate \
#                 presidio-analyzer presidio-anonymizer scikit-learn spacy
# !python -m spacy download en_core_web_lg
# !python -m spacy download en_core_web_sm


## Setup


In [2]:
import sys
sys.path.insert(0, "../src")

import time
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from anonymisation.data import load_tab
from anonymisation.evaluation import (
    evaluate_document, merge_results, results_to_dataframe,
)
from anonymisation.predictors import make_hf_predictor, DEFAULT_HF_LABEL_TO_TAB
from anonymisation.device import best_device, report_device

print(report_device())


Device: mps    (Apple Silicon (MPS))


## Configuration


In [3]:
HF_MODEL = "dslim/bert-base-NER"
USE_FULL_TEST_SET = True
SAMPLE_SIZE = 100              # only used when USE_FULL_TEST_SET = False
MATCH_MODES = ["partial", "exact"]
RESULTS_PATH = "../results/baseline_huggingface.csv"


## Load the model


In [4]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

device, _ = best_device()
print(f"Loading {HF_MODEL} on device={device} ...")
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
model = AutoModelForTokenClassification.from_pretrained(HF_MODEL)

# device for HF pipeline: -1 = CPU, 0 = first CUDA. MPS support in pipeline is
# patchy across transformers versions, so we fall back to CPU on MPS.
hf_device = 0 if device == "cuda" else -1
ner = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=hf_device,
)
predict = make_hf_predictor(ner)

# Smoke test
out = predict("Maria Petrova lives in Sofia.")
print("Sanity check:", out)


Loading dslim/bert-base-NER on device=mps ...


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Sanity check: [(0, 12, 'PERSON', 'Maria Petrov'), (23, 28, 'LOC', 'Sofia')]


## Load TAB & run the evaluation


In [5]:
dataset = load_tab()
test_docs = list(dataset["test"])
if not USE_FULL_TEST_SET:
    test_docs = test_docs[:SAMPLE_SIZE]
print(f"Evaluating on {len(test_docs)} TAB test documents.")


Evaluating on 555 TAB test documents.


In [6]:
all_merged = {}
for mode in MATCH_MODES:
    print(f"\n--- {mode} match ---")
    per_doc = []
    start = time.time()
    for i, doc in enumerate(test_docs):
        if (i + 1) % 50 == 0:
            elapsed = time.time() - start
            print(f"  {i + 1}/{len(test_docs)}   ({(i + 1)/elapsed:.1f} docs/s)")
        per_doc.append(evaluate_document(predict, doc, mode=mode))
    elapsed = time.time() - start
    print(f"  done in {elapsed:.1f}s")
    all_merged[mode] = merge_results(per_doc)

results_df = results_to_dataframe(all_merged)
results_df.insert(0, "model", "hf_bert_base_ner")
results_df.to_csv(RESULTS_PATH, index=False)
print(f"\nSaved → {RESULTS_PATH}")



--- partial match ---
  50/555   (9.2 docs/s)
  100/555   (9.1 docs/s)
  150/555   (8.8 docs/s)
  200/555   (8.9 docs/s)
  250/555   (9.0 docs/s)
  300/555   (9.0 docs/s)
  350/555   (9.1 docs/s)
  400/555   (9.3 docs/s)
  450/555   (9.3 docs/s)
  500/555   (9.2 docs/s)
  550/555   (9.1 docs/s)
  done in 60.8s

--- exact match ---
  50/555   (9.0 docs/s)
  100/555   (9.1 docs/s)
  150/555   (8.8 docs/s)
  200/555   (8.8 docs/s)
  250/555   (8.9 docs/s)
  300/555   (9.0 docs/s)
  350/555   (9.2 docs/s)
  400/555   (9.4 docs/s)
  450/555   (9.2 docs/s)
  500/555   (8.7 docs/s)
  550/555   (8.4 docs/s)
  done in 66.1s

Saved → results/hf_results.csv


## Results table


In [7]:
from anonymisation.mapping import TAB_TO_SPACY

for mode in MATCH_MODES:
    merged = all_merged[mode]
    print(f"\n── {mode.upper()} MATCH ──")
    rows = []
    for et in list(TAB_TO_SPACY.keys()) + ["_ALL"]:
        r = merged[et]
        rows.append({
            "Entity": et if et != "_ALL" else "▶ OVERALL",
            "TP": r.tp, "FP": r.fp, "FN": r.fn,
            "Precision": f"{r.precision:.1%}",
            "Recall":    f"{r.recall:.1%}",
            "F1":        f"{r.f1:.1%}",
        })
    print(pd.DataFrame(rows).to_string(index=False))



── PARTIAL MATCH ──
   Entity   TP    FP    FN Precision Recall    F1
   PERSON 1999  1987  2139     50.2%  48.3% 49.2%
      ORG  476  6717  1470      6.6%  24.5% 10.4%
      LOC  828  2512   631     24.8%  56.8% 34.5%
 DATETIME    0     0  9536      0.0%   0.0%  0.0%
 QUANTITY    0     0   664      0.0%   0.0%  0.0%
     CODE    0     0  1612      0.0%   0.0%  0.0%
      DEM    0     0   917      0.0%   0.0%  0.0%
     MISC   30  4586   507      0.6%   5.6%  1.2%
▶ OVERALL 3333 15802 17476     17.4%  16.0% 16.7%

── EXACT MATCH ──
   Entity   TP    FP    FN Precision Recall    F1
   PERSON  112  3874  4026      2.8%   2.7%  2.8%
      ORG  236  6957  1710      3.3%  12.1%  5.2%
      LOC  737  2603   722     22.1%  50.5% 30.7%
 DATETIME    0     0  9536      0.0%   0.0%  0.0%
 QUANTITY    0     0   664      0.0%   0.0%  0.0%
     CODE    0     0  1612      0.0%   0.0%  0.0%
      DEM    0     0   917      0.0%   0.0%  0.0%
     MISC    5  4611   532      0.1%   0.9%  0.2%
▶ OVERALL 

## What to look for in the numbers

The CoNLL-2003 label set covers `PER / ORG / LOC / MISC`. Anything in TAB that doesn't map to one of these — `DATETIME`, `QUANTITY`, `CODE`, `DEM` — is **structurally invisible** to this model, so we expect 0% recall on those four entity types.

This is an instructive failure mode rather than a surprising one: it confirms that the gap we saw in Phase 1 is not a quirk of spaCy. Any general-purpose English NER will hit a wall on legal-domain categories. To close that gap you have to *train on the right labels* — which is what notebook 03 does.

The `MISC` category is also worth noting: HF's MISC will fire on a much wider population than TAB's MISC (works of art, events, products), which means we should expect high false-positive rates there.

We'll combine these results with Phase 1's spaCy numbers in `04_head_to_head.ipynb`.
